In [ ]:
using Plots
using Revise
using Apr30Project
using DifferentialEquations
using OrdinaryDiffEqLowOrderRK  # needed for Euler
using OrdinaryDiffEqSDIRK       # needed for ImplicitEuler

# Advection Equation

## Euler and Implicit Euler

In [ ]:
# set a common problem
nx = 1000;

x = LinRange(-5,5, nx+1)[2:end]; # skip the first point because we are not solving there
uinit = @. exp(-x^2);
c = 1.0;
dx = x[2] - x[1];
# g = t->cos(pi *t);
# f = (x,t) -> 0.1 * sin(pi * x);

g = t->0;
f = (x,t) ->0;

tspan = (0.0, 5.0);

Dx = sparse_backwards_difference_matrix(nx, dx);

# store the parameters in a tuple structure:
p = (c, dx, x, g, f, Dx);

prob = ODEProblem(advection!, uinit, tspan, p);

In [ ]:
sol = solve(prob); # automatic solver

dt = 0.1;
sol_euler = solve(prob, Euler(), dt=dt, adaptive=false);
sol_impeeuler = solve(prob, ImplicitEuler(), dt=dt, adaptive=false);

In [ ]:
t_plt = LinRange(tspan[1], tspan[2], 101);
anim = @animate for i in 1:length(t_plt)
    plot(x, sol(t_plt[i]), title = "t = $(round(t_plt[i], digits=2))", ylim=(-1,1),label="Auto")
    plot!(x, sol_euler(t_plt[i]), title = "t = $(round(t_plt[i], digits=2))", ylim=(-1,1),label="Euler")
    plot!(x, sol_impeeuler(t_plt[i]), title = "t = $(round(t_plt[i], digits=2))", ylim=(-1,1),label="Implicit Euler")
    xlabel!("x")
    ylabel!("u(x,t)")
end
gif(anim, "advection.gif", fps=3)

* Sometimes classical (i.e., Euler, Implicit Euler) work better than fancy adaptive methods
* Worth trying different options (i.e., different solvers)
* Something bad happens for Euler when the time step gets too big.

## Sparse Matrices (CSC Format)

In [ ]:
nx = 5;

x = LinRange(-5,5, nx+1)[2:end]; # skip the first point because we are not solving there
dx = x[2] - x[1];
Dx = sparse_backwards_difference_matrix(nx, dx);
Dx

Examine the elements of the CSC data structure

In [ ]:
Dx.m # number of rows

In [ ]:
Dx.n # number of columns

In [ ]:
Dx.nzval # array of the nonzero values in the matrix

In [ ]:
Dx.rowval # corresponding rows of the entries from Dx.nzval

In [ ]:
Dx.colptr

In [ ]:
for j in 1:Dx.n
    for k in Dx.colptr[j]:(Dx.colptr[j+1]-1)
        i = Dx.rowval[k]
        v = Dx.nzval[k]
        println("Matrix[$(i), $(j)] = $(round(v, digits=2))")
    end
end